In [107]:
import langchain

In [108]:
import os
from typing import List, Dict, Any
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

In [109]:
from langchain_core.documents import Document
from langchain_text_splitters import (
	RecursiveCharacterTextSplitter,
	CharacterTextSplitter,
	TokenTextSplitter
)

#### Understanding Document Structure in Langchain

In [110]:
# create a simple document 

doc = Document(
	page_content = "This is the main text content that will be embedded and searched.",
	metadata = {
		"source": "example.txt",
		"page": 1,
		"author": "Dilli Ram",
		"date_created": "2026-8-01",
		"custom_field": "any_value"
	}
)

print("Document structure")
print(f"content: ", doc.page_content)
print(f"content: ", doc.metadata)

Document structure
content:  This is the main text content that will be embedded and searched.
content:  {'source': 'example.txt', 'page': 1, 'author': 'Dilli Ram', 'date_created': '2026-8-01', 'custom_field': 'any_value'}


##### Why metadata matters!

In [111]:
print("\nMetadata is crucial for: ")
print("- Filtering search results.")
print("- Tracking document sources.")
print("- Providing context in responses.")
print("- Debugging and auditing.")


Metadata is crucial for: 
- Filtering search results.
- Tracking document sources.
- Providing context in responses.
- Debugging and auditing.


In [112]:
type(doc)

langchain_core.documents.base.Document

#### Self practice

In [113]:
from langchain_core.documents import Document

doc = Document(
	page_content= """
    University Attendance Policy

    Students must maintain at least 80% attendance.
    Students below 80% attendance may not be allowed to sit for examinations.
    """,
  metadata = {
				"source": "university_policy.txt",
        "department": "Academic",
        "year": 2026
	}
)

print(doc.page_content)
print(doc.metadata)


    University Attendance Policy

    Students must maintain at least 80% attendance.
    Students below 80% attendance may not be allowed to sit for examinations.
    
{'source': 'university_policy.txt', 'department': 'Academic', 'year': 2026}


In [114]:
# multiple documents

from langchain_core.documents import Document

documents = [
  Document(
        page_content="Python is a high-level programming language.",
        metadata={"source": "python.txt", "topic": "programming"}
    ),
  Document(
        page_content="LangChain is a framework for building applications with LLMs.",
        metadata={"source": "langchain.txt", "topic": "AI"}
    ),
  Document(
        page_content="Transformers use self-attention to process sequences.",
        metadata={"source": "transformers.txt", "topic": "deep learning"}
    )
]

In [115]:
print(len(documents))

3


In [116]:
print(documents[0])

page_content='Python is a high-level programming language.' metadata={'source': 'python.txt', 'topic': 'programming'}


In [117]:
print(documents[0].page_content)
print(documents[0].metadata)

Python is a high-level programming language.
{'source': 'python.txt', 'topic': 'programming'}


##### Inspecting document structure

In [118]:
for i, doc in enumerate(documents):
  print(f"Document:", i)
  print(f"Content: ", doc.page_content)
  print(f"Metadata: ", doc.metadata)
  print("-"*75)

Document: 0
Content:  Python is a high-level programming language.
Metadata:  {'source': 'python.txt', 'topic': 'programming'}
---------------------------------------------------------------------------
Document: 1
Content:  LangChain is a framework for building applications with LLMs.
Metadata:  {'source': 'langchain.txt', 'topic': 'AI'}
---------------------------------------------------------------------------
Document: 2
Content:  Transformers use self-attention to process sequences.
Metadata:  {'source': 'transformers.txt', 'topic': 'deep learning'}
---------------------------------------------------------------------------


##### Text files (.txt) - The Simplest Case (#2 -text-files)

In [119]:
import os 

os.makedirs("data/text_files", exist_ok=True)

In [120]:
sample_texts = {
	"data/text_files/python_intro.txt": """
Python programming is an easy-to-learn, high-level computer language used for web apps, data analysis, and automation.Core Python ConceptsVariables: Containers used to store data values in memory.Data Types: Basic classifications for data, such as numbers (integers and floats) and strings (text).Functions: Reusable blocks of code that perform specific tasks, like the built-in print() command.Collections: Structures like lists and dictionaries used to group multiple items together.Loops: Control structures like for and while used to repeat actions multiple times.
""",
	"data/text_files/machine_learning.txt": """
Machine learning (ML) is a branch of artificial intelligence where computers learn patterns from data to make decisions without being explicitly programmed. Instead of writing rigid rules, you train a model on historical data so it can make predictions on new, unseen information.Core Machine Learning ConceptsSupervised Learning: Training a model on labeled data, where the correct answers are already known (e.g., predicting house prices).Unsupervised Learning: Finding hidden patterns or groupings in unlabeled data (e.g., segmenting customers into different buying behaviors).Reinforcement Learning: Teaching an agent to make decisions through a system of rewards and punishments (e.g., training an AI to play chess).
"""
}

In [121]:
for filepath, content in sample_texts.items():
  with open(filepath, 'w', encoding='utf-8') as f:
    f.write(content)

print("✅ sample file created.")

✅ sample file created.


#### TextLoader - Read single file

In [122]:
from langchain_community.document_loaders import TextLoader

# laoding a single file
loader = TextLoader("data/text_files/python_intro.txt", encoding='utf-8')
loader

In [123]:
documents = loader.load()
print(type(documents))
print(documents)

<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='\nPython programming is an easy-to-learn, high-level computer language used for web apps, data analysis, and automation.Core Python ConceptsVariables: Containers used to store data values in memory.Data Types: Basic classifications for data, such as numbers (integers and floats) and strings (text).Functions: Reusable blocks of code that perform specific tasks, like the built-in print() command.Collections: Structures like lists and dictionaries used to group multiple items together.Loops: Control structures like for and while used to repeat actions multiple times.\n')]


In [124]:
print(f"Loaded {len(documents)} documents...")
print(f"Document preview: {documents[0].page_content[:100]}")
print(f"Metadata document: {documents[0].metadata}")

Loaded 1 documents...
Document preview: 
Python programming is an easy-to-learn, high-level computer language used for web apps, data analys
Metadata document: {'source': 'data/text_files/python_intro.txt'}


#### Directory Loader - Multiple Text files

In [125]:
from langchain_community.document_loaders import DirectoryLoader

# load all the text files from the directory
dir_loader = DirectoryLoader(
	"data/text_files",
	glob="**/*.txt", 							# pattern to match the files
	loader_kwargs= {'encoding': 'utf-8'},
	show_progress=True
)

documents = dir_loader.load()

print(f"Loaded {len(documents)} documents...")
for i, doc in enumerate(documents):
  print(f"\nDocument {i+1}")
  print(f"source: {doc.metadata["source"]}")
  print(f"Length: {len(doc.page_content)} characters...")

  0%|          | 0/2 [00:00<?, ?it/s]libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
100%|██████████| 2/2 [00:00<00:00, 19.85it/s]

Loaded 2 documents...

Document 1
source: data/text_files/python_intro.txt
Length: 568 characters...

Document 2
source: data/text_files/machine_learning.txt
Length: 721 characters...


#### Loading PDFs

In [126]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/text_files/mypdf.pdf")
documents = loader.load()
print(documents)
print(len(documents))

[Document(metadata={'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'PyPDF', 'creationdate': '2023-08-03T21:32:56+00:00', 'meeting starting date': '19 June 2023', 'moddate': '2023-09-30T08:26:56-04:00', 'ieee article id': '10266218', 'ieee issue id': '10265772', 'subject': '2023 3rd International Conference on Pervasive Computing and Social Networking (ICPCSN);2023; ; ;10.1109/ICPCSN58827.2023.00028', 'ieee publication id': '10265860', 'title': 'An ANPR-Based Automatic Toll Tax Collection System Using Camera', 'meeting ending date': '20 June 2023', 'source': 'data/text_files/mypdf.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}, page_content="An ANPR-based Automatic Toll Tax Collection System \nusing Camera \n \nB VeerasekharReddy  \nInformation Technology \nMLR Institute of Technology \nHyderabad, India \nbhargavisekhar68@gmail.com \n \nSahithi Sindhu Gadup

In [127]:
for doc in documents:
  print(doc.metadata)

{'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'PyPDF', 'creationdate': '2023-08-03T21:32:56+00:00', 'meeting starting date': '19 June 2023', 'moddate': '2023-09-30T08:26:56-04:00', 'ieee article id': '10266218', 'ieee issue id': '10265772', 'subject': '2023 3rd International Conference on Pervasive Computing and Social Networking (ICPCSN);2023; ; ;10.1109/ICPCSN58827.2023.00028', 'ieee publication id': '10265860', 'title': 'An ANPR-Based Automatic Toll Tax Collection System Using Camera', 'meeting ending date': '20 June 2023', 'source': 'data/text_files/mypdf.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}
{'producer': 'OpenPDF 1.0.0-SNAPSHOT; modified using iTextSharp 5.0.2 (c) 1T3XT BVBA; modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'PyPDF', 'creationdate': '2023-08-03T21:32:56+00:00', 'meeting starting date': '

##### Loading Word documents

In [128]:
import docx

doc = docx.Document()
doc.add_paragraph("I live in Nepal and this country is very beautiful located in the lap of the Himalayas.")
doc.save("data/text_files/mydoc.docx")

In [129]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader("data/text_files/mydoc.docx")
documents = loader.load()
print(documents[0].page_content)

I live in Nepal and this country is very beautiful located in the lap of the Himalayas.


##### The code below is re-rerun so that I can implement the text splitting strategies

In [130]:
from langchain_community.document_loaders import DirectoryLoader

# load all the text files from the directory
dir_loader = DirectoryLoader(
	"data/text_files",
	glob="**/*.txt", 							# pattern to match the files
	loader_kwargs= {'encoding': 'utf-8'},
	show_progress=True
)

documents = dir_loader.load()

  0%|          | 0/2 [00:00<?, ?it/s]libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
100%|██████████| 2/2 [00:00<00:00, 215.24it/s]


#### Text splitting strategies

In [131]:
from langchain_text_splitters import (
	CharacterTextSplitter,
	RecursiveCharacterTextSplitter,
	TokenTextSplitter
)

print(documents)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python programming is an easy-to-learn, high-level computer language used for web apps, data analysis, and automation.Core Python ConceptsVariables: Containers used to store data values in memory.Data Types: Basic classifications for data, such as numbers (integers and floats) and strings (text).Functions: Reusable blocks of code that perform specific tasks, like the built-in print() command.Collections: Structures like lists and dictionaries used to group multiple items together.Loops: Control structures like for and while used to repeat actions multiple times.'), Document(metadata={'source': 'data/text_files/machine_learning.txt'}, page_content='Machine learning (ML) is a branch of artificial intelligence where computers learn patterns from data to make decisions without being explicitly programmed. Instead of writing rigid rules, you train a model on historical data so it can make predictions on new, un

##### Character text splitter - 1

In [132]:
print("Character text splitter: ")
char_splitter = CharacterTextSplitter(
	separator="\n", 			# split on newline
	chunk_size=200, 			# max chunk size in characters
	chunk_overlap=20, 		# overlap between chunks
	length_function=len		# how to measure chunk size
)

Character text splitter: 


In [133]:
text = documents[0].page_content	
text 

'Python programming is an easy-to-learn, high-level computer language used for web apps, data analysis, and automation.Core Python ConceptsVariables: Containers used to store data values in memory.Data Types: Basic classifications for data, such as numbers (integers and floats) and strings (text).Functions: Reusable blocks of code that perform specific tasks, like the built-in print() command.Collections: Structures like lists and dictionaries used to group multiple items together.Loops: Control structures like for and while used to repeat actions multiple times.'

In [134]:
char_chunks = char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

Created 1 chunks
First chunk: Python programming is an easy-to-learn, high-level computer language used for web apps, data analysi...


##### One more example

In [136]:
from langchain_text_splitters import CharacterTextSplitter

text = """
Artificial intelligence is a field of computer science.
Machine learning is a subset of artificial intelligence.
Deep learning is a subset of machine learning.
"""

splitter = CharacterTextSplitter(
	separator='\n', 
	chunk_size=100,
	chunk_overlap=20
)

chunks = splitter.split_text(text)
for i, chunk in enumerate(chunks):
  print(f"------------------- chunk {i+1} ----------------")
  print(chunk)

------------------- chunk 1 ----------------
Artificial intelligence is a field of computer science.
------------------- chunk 2 ----------------
Machine learning is a subset of artificial intelligence.
------------------- chunk 3 ----------------
Deep learning is a subset of machine learning.


##### Character text splitter - 2
- Recursive Character Text Splitter

In [137]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = """"
Artificial intelligence is transforming many industries.

Machine learning allows computers to learn patterns
from data without being explicitly programmed.

Deep learning uses neural networks with multiple
layers to learn complex representations.
"""

splitter = RecursiveCharacterTextSplitter(
	chunk_size=200,
	chunk_overlap=40,
)

chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
  print(f"\n----------- chunk {i+1} -------------")
  print(chunk)


----------- chunk 1 -------------
"
Artificial intelligence is transforming many industries.

Machine learning allows computers to learn patterns
from data without being explicitly programmed.

----------- chunk 2 -------------
Deep learning uses neural networks with multiple
layers to learn complex representations.


##### Splitting documents instead of strings

In [139]:
from langchain_core.documents import Document

documents = [
	Document(page_content="""
          Artificial intelligence is a field of computer science.
          Machine learning is a subset of artificial intelligence.
          """,
          metadata={
						"source": "ai_notes.txt",
						"page": 1
					})
]

In [145]:
splitter = RecursiveCharacterTextSplitter(
	chunk_size=50,
	chunk_overlap=30
)

chunks = splitter.split_documents(documents)

In [146]:
type(chunks)

list

In [147]:
for i, chunk in enumerate(chunks):
  print(f"--------chunk {i+1}--------")
  print(chunk)

--------chunk 1--------
page_content='Artificial intelligence is a field of' metadata={'source': 'ai_notes.txt', 'page': 1}
--------chunk 2--------
page_content='intelligence is a field of computer science.' metadata={'source': 'ai_notes.txt', 'page': 1}
--------chunk 3--------
page_content='Machine learning is a subset of' metadata={'source': 'ai_notes.txt', 'page': 1}
--------chunk 4--------
page_content='learning is a subset of artificial intelligence.' metadata={'source': 'ai_notes.txt', 'page': 1}


In [151]:
os.makedirs("data/text_files", exist_ok=True)
with open("data/text_files/data.txt", "w") as f:
  f.write("""IThe Federal Democratic Republic of Nepal is a landlocked nation nestled in the heart of South Asia, majorly positioned along the majestic Himalayan mountain range. Bordered by the Tibet Autonomous Region of China to the north and surrounded by India to the south, east, and west, Nepal occupies a unique geopolitical, cultural, and ecological niche on the global map. Covering an area of approximately 147,516 square kilometers, this sovereign state stretches roughly 885 kilometers from east to west and between 145 to 241 kilometers from north to south. Despite its relatively compact territorial footprint, Nepal exhibits some of the sharpest geographical and altitudinal variations found anywhere on Earth. The country descends from the world's highest peak, Mount Everest (Sagarmatha), towering at 8,848.86 meters above sea level, down to the subtropical alluvial plains of the Terai region, which sit barely 60 meters above sea level. This profound topographical gradient yields an astonishing diversity of ecosystems, microclimates, and biomes, compressed into a narrow geographical corridor.

Nepal’s geographical identity is defined by three distinct parallel ecological belts running from east to west: the Mountain Region (Himal), the Hill Region (Pahad), and the Plain Region (Terai). Each of these zones contributes uniquely to the country's ecological balance, agricultural productivity, settlement patterns, and cultural fabric. 

The Mountain Region comprises the northernmost spine of the country, constituting about 35 percent of Nepal's total land area. This zone features more than 200 peaks exceeding 6,000 meters in elevation, including eight of the world's fourteen independent mountain summits above 8,000 meters. Beyond Mount Everest, these giants include Kanchenjunga, Lhotse, Makalu, Cho Oyu, Dhaulagiri, Manaslu, and Annapurna. This region is characterized by harsh alpine climates, sub-zero temperatures, permanent snowlines, glaciers, and rugged terrain. Due to these inhospitable conditions, the mountain belt is sparsely populated, inhabited primarily by indigenous communities such as the Sherpas, Bhutias, and Thakalis, who have adapted over centuries to high-altitude pastoralism, trans-Himalayan trade, and more recently, mountaineering tourism.

South of the high mountains lies the Hill Region, encompassing roughly 42 percent of the country’s territory. Ranging from 600 to 3,000 meters in altitude, this region features the Mahabharat Lekh (Lesser Himalaya) and the gentler Churia (Sivalik) hills. The Hill Region is dissected by deep river valleys and tectonic basins, most notably the Kathmandu Valley and the Pokhara Valley, which serve as historical hubs of urban civilization, politics, art, and commerce. The climate here varies from subtropical at lower elevations to temperate at higher ridges, making it highly conducive to human settlement and terrace farming. For generations, the hills have formed the cultural and political core of modern Nepal, home to a tapestry of ethnic groups including the Khas-Arya (Bahun, Chhetri), Newars, Gurungs, Magars, Tamangs, Rais, and Limbus.

The southernmost strip of Nepal is the Terai, a low-lying northern extension of the Indo-Gangetic Plain. Accounting for about 23 percent of the country's land area, the Terai features fertile alluvial soil, a dense network of meandering rivers, and a hot, humid tropical or subtropical climate. Historically covered by dense, malaria-ridden forests known as the 'Char Kose Jhadi', the Terai underwent major demographic shifts in the mid-20th century following malaria eradication programs. Today, it has transformed into the 'granary of Nepal', generating the vast majority of the nation's food supplies and hosting more than half of the total population. The Terai is culturally distinct, characterized by deep-rooted ties to the adjacent plains of India, and is inhabited by communities such as the Tharus, Maithils, Bhojpuris, and Madhesis, along with millions of hill migrants who settled there in recent decades.

Nepal's hydrology is dominated by three major perennial river systems, all originating from the snowmelt of the Himalayas or the Tibetan plateau: the Koshi system in the east, the Gandaki system in the central region, and the Karnali system in the west. These rivers cut deep gorges through the mountains and hills, flowing south to converge eventually with the Ganges in India. These water resources grant Nepal an immense, though largely untapped, potential for hydroelectricity generation and irrigation. The country is also decorated with beautiful alpine and glacial lakes, such as Rara Lake (the largest), Shey Phoksundo Lake (the deepest), Tilicho Lake (one of the highest lakes globally), and Fewa Lake in Pokhara, which are critical tourist attractions and ecological sanctuaries.

The historical narrative of Nepal is a complex mosaic of indigenous developments, cross-cultural exchanges, dynastic transformations, and a fierce commitment to national sovereignty. Archaeological evidence, including Neolithic tools found in the Kathmandu Valley, indicates that human habitation in the Himalayan foothills dates back at least 11,000 years. The earliest recorded history of Nepal is closely intertwined with the Kathmandu Valley, which was anciently referred to simply as 'Nepal'.

The Kirats are recognized as the first prominent ruling dynasty of the Kathmandu Valley, documented in ancient Hindu texts like the Mahabharata. Believed to have ruled from around the 8th century BCE to the 30th century CE, the Kirats were known for their sophisticated governance, agricultural focus, and trade relations. King Yalamber is celebrated as the first Kirat monarch. It was during the Kirat era that Buddhism arrived in Nepal, traditionally marked by the historic visit of the Indian Emperor Ashoka to the valley in the 3rd century BCE, where he erected four famous stupas in Patan (Lalitpur) and left an inscribed pillar at Lumbini, the birthplace of Gautama Buddha.

Following the decline of the Kirats, the Licchavi Dynasty emerged around the 4th century CE. Originating from northern India, the Licchavis introduced classical Sanskrit culture, Vaishnavite and Shaivite Hinduism, and advanced administrative structures. The Licchavi period is widely lauded as the 'Golden Age' of Nepal’s history. Under illustrious rulers like King Manadeva I (who built the Changu Narayan Temple and issued the first dated inscriptions) and later King Amsuverma (who established strategic matrimonial and diplomatic alliances with Tibet and China), Nepal flourished as a powerful mercantile center controlling trans-Himalayan trade routes. Art, architecture, sculpture, and religious harmony reached unparalleled heights.

By the 13th century, the Malla Dynasty seized control of the Kathmandu Valley, ushering in an era of extraordinary artistic and urban renaissance. The Mallas restructured society based on orthodox Hindu principles but remained deeply patrons of the arts. Under King Jayasthiti Malla, code of laws and social systems were codified. In the 15th century, King Yaksha Malla divided the kingdom among his three sons, splitting the Kathmandu Valley into three rival city-states: Kathmandu (Kantipur), Patan (Lalitpur), and Bhaktapur (Bhadgaon). This fragmentation led to intense artistic rivalry; each king sought to construct more magnificent palace squares, temples, and monuments than his neighbors. This competitive patronage produced the iconic Durbar Squares that today stand as UNESCO World Heritage sites, showcasing distinct Newari pagoda architecture, intricate wood carvings, and masterfully cast bronze structures. Outside the valley, the rest of modern Nepal was divided into dozens of petty principalities: the Baise Rajya (22 kingdoms) in the Karnali region and the Chaubise Rajya (24 kingdoms) in the Gandaki region, alongside various eastern Kirat principalities.

The modern history of Nepal began in the mid-18th century with the visionary and ambitious campaign of Prithvi Narayan Shah, the ruler of the small hill kingdom of Gorkha. Recognizing the vulnerability of a fractured Himalayan region to the expanding colonial might of the British East India Company, Prithvi Narayan Shah embarked on a monumental campaign to unify the fragmented principalities. Through strategic warfare, economic blockades, and masterful diplomacy, the Gorkhali forces captured the strategic hillfort of Nuwakot, followed by the dramatic conquest of Kathmandu, Patan, and Bhaktapur between 1768 and 1769. He shifted his capital to Kathmandu, founding the Shah Dynasty of a unified Nepal. His successors expanded the empire’s borders aggressively, stretching from the Teesta River in the east to the Kangra Valley (modern Himachal Pradesh, India) in the west.

This rapid expansion brought the young Gorkha Empire into direct conflict with the British East India Company, culminating in the Anglo-Nepalese War (1814–1816). Despite displaying legendary bravery—which earned the Nepali soldiers the reputation of fierce 'Gurkhas'—Nepal was forced to sign the Treaty of Sugauli in 1816. The treaty forced Nepal to surrender roughly one-third of its territory, including Sikkim, Kumaon, Garhwal, and large parts of the Terai (some of which were later restored), establishing the country’s modern frontiers. Crucially, Nepal managed to preserve its independence, never falling under formal European colonial rule—a distinct source of pride for its citizens.

Internal political instability plagued the Shah court following the war, characterized by bloody factions, assassinations, and weak monarchs. This instability culminated in the infamous Kot Massacre of September 14, 1846, orchestrated by an ambitious army general named Jung Bahadur Rana. Jung Bahadur eliminated all political rivals, stripped the Shah king of executive authority, and appointed himself Prime Minister, establishing a hereditary autocracy. For the next 104 years, the Rana Dynasty ruled Nepal with absolute power. The Ranas pursued a strict policy of isolationism, keeping the country closed to foreigners to prevent colonial subversion while maintaining cordial ties with the British Empire, regularly supplying Gurkha soldiers to fight in British imperial campaigns, including World War I and World War II. While the Rana era brought some infrastructure, like the country’s first modern school (Durbar High School) and hospital (Bir Hospital), it left the wider populace illiterate, impoverished, and economically stagnant.

The tide turned after World War II, fueled by anti-colonial movements in neighboring India and growing domestic dissent. In 1950, a popular revolution led by the newly formed Nepali Congress party, with the strategic backing of King Tribhuvan (who fled the palace to seek asylum in India), overthrew the Rana regime. In early 1951, King Tribhuvan returned to Kathmandu, proclaiming the dawn of democracy. However, the democratic experiment was short-lived. Following a brief period of multi-party governance, King Mahendra (Tribhuvan’s successor) staged a royal coup in 1960, dissolving the democratically elected government of B.P. Koirala, banning political parties, and introducing the party-less Panchayat system. For thirty years, this absolute monarchical system governed Nepal under the banner of nationalist modernization, but faced growing underground resistance.

In 1990, the Jan Andolan (People's Movement) forced King Birendra to abolish the Panchayat system and restore multi-party democracy under a constitutional monarchy. Yet, political instability persisted, with frequent changes in government. In 1996, the Communist Party of Nepal (Maoist) launched a violent insurgency, aiming to overthrow the monarchy and establish a people’s republic. This civil war lasted for a decade, claiming over 17,000 lives, displacing thousands, and severely disrupting the economy. The nation suffered an additional shock on June 1, 2001, during the tragic Royal Massacre, where Crown Prince Dipendra reportedly shot and killed King Birendra, Queen Aishwarya, and several members of the royal family before shooting himself. King Gyanendra, Birendra's brother, ascended the throne and later assumed absolute power in 2005 to crush the insurgency, a move that backfired.

Gyanendra's authoritarian actions united mainstream political parties and the Maoist rebels against the crown. The historic Jan Andolan II (Second People's Movement) in April 2006 filled the streets of Kathmandu with millions of protestors, forcing the king to reinstate parliament. A Comprehensive Peace Accord was signed later that year, officially ending the civil war and bringing the Maoists into the political mainstream. In May 2008, the newly elected Constituent Assembly took the historic step of formally abolishing the 240-year-old monarchy, declaring Nepal a Federal Democratic Republic. After years of delicate transitions and the devastation of the catastrophic April 2015 earthquake, Nepal adopted a progressive new Constitution on September 20, 2015, institutionalizing a three-tiered federal system of governance comprising federal, provincial, and local levels.

Nepal’s socio-cultural matrix is characterized by staggering diversity, earning it the metaphorical description of "a garden of four varnas and thirty-six castes" coined by its founder, Prithvi Narayan Shah. According to official national census records, Nepal is home to over 142 distinct ethnic and caste groups speaking more than 124 mother tongues. This multiculturalism has evolved over millennia, shaped by successive migration waves from Central Asia, Tibet, and the Indo-Gangetic plains, meeting in the fertile mid-hills.

Religion is the foundational cornerstone of daily life in Nepal. Formerly the world's only official Hindu Kingdom, Nepal is now constitutionally a secular state, though Hinduism remains the dominant religion, practiced by over 80 percent of the population. However, Hinduism in Nepal is uniquely syncretic, heavily intertwined with Buddhism and indigenous animist traditions. Statues of Hindu deities and Buddhist Bodhisattvas sit side-by-side in ancient temples, and members of both faiths participate enthusiastically in each other’s major festivals. The Kathmandu Valley's Newari culture is a prime example of this hybridity, where unique forms of Vajrayana Buddhism incorporate Hindu caste systems and deities.

Buddhism, followed by roughly 9 percent of the population, holds deep historical roots. Lumbini, located in the southwestern plains of Nepal, is globally revered as the holy site where Queen Mayadevi gave birth to Siddhartha Gautama around 563 BCE. Today, Lumbini is a UNESCO World Heritage site and a global center for pilgrimage, featuring international monasteries, peace pagodas, and ancient ruins. Across the hills and mountains, Tibetan-influenced Mahayana Buddhism is practiced extensively by Sherpas, Tamangs, Gurungs, and Tibetans. Iconic monuments like the Swayambhunath Stupa (the Monkey Temple) and the colossal Boudhanath Stupa stand as glowing spiritual beacons dominant in the Kathmandu skyline. Islam, Christianity, Kirat (an ancient nature-worshiping religion practiced by eastern ethnic groups), and Bon traditions also contribute to the nation’s diverse spiritual fabric.

Linguistically, Nepal is highly pluralistic. Nepali, an Indo-Aryan language written in the Devanagari script, serves as the official national language and the common lingua franca connecting different ethnic groups. Major regional and ethnic languages include Maithili, Bhojpuri, and Tharu spoken widely in the Terai region, alongside Nepal Bhasa (Newari), Tamang, Magar, Gurung, Rai, Limbu, and Sherpa, belonging predominantly to the Tibeto-Burman language family.

Festivals are a central element of Nepali culture, occurring almost continuously throughout the year. The grandest national festival is Dashain, a fifteen-day celebration observed by Hindus across the country to commemorate the victory of Goddess Durga over the demon Mahishasura, symbolizing the triumph of good over evil. Dashain is marked by family reunions, receiving blessings and 'Tika' from elders, flying kites, and animal sacrifices. This is followed shortly by Tihar (Dipawali), the festival of lights, which uniquely honors animals like crows, dogs, cows, and oxen on consecutive days, culminating in Bhai Tika, a celebration of the sacred bond between brothers and sisters. Other major festivals include Chhath, celebrated with immense fervor in the Terai with rituals dedicated to the Sun God; Maha Shivaratri, drawing hundreds of thousands of Hindu pilgrims and holy sadhus from across South Asia to the sacred Pashupatinath Temple; Lhosar, the Buddhist New Year celebrated by various high-altitude ethnic groups; Holi, the vibrant spring festival of colors; and Gai Jatra, a Newari festival honoring deceased family members with humor and satire.

Art and architecture in Nepal are traditionally religious in nature. The Newar craftsmanship of the Kathmandu Valley represents the pinnacle of traditional Nepali art. The unique multi-roofed pagoda style of temple architecture, which later influenced building styles across East Asia, was perfected here. These structures feature beautifully carved wooden struts, windows (such as the famous Peacock Window in Bhaktapur), and intricate toranas above gilded doorways. The art of Paubha painting (traditional religious scrolls similar to Tibetan Thangkas), bronze casting, and pottery have been passed down through generations of artisans and remain vital economic and cultural enterprises.

Culinary traditions reflect the country's geography and diverse ethnic influences. The unofficial national dish of Nepal is Dal Bhat Tarkari—a wholesome, balanced meal consisting of steamed rice (Bhat), lentil soup (Dal), and seasonally spiced vegetable curries (Tarkari), often accompanied by spicy pickles (Achar) and meat. Eaten twice daily by a vast majority of Nepalis, it provides sustained energy for arduous manual labor in the hills. In urban areas and globally, Momo—steamed or fried dumplings filled with minced meat or vegetables and served with a tangy tomato-sesame sauce—has attained the status of a culinary cultural icon. Other regional specialties include Dhido (a dense, nutritious porridge made from buckwheat or millet, common in rural hills), Selroti (a ring-shaped, deep-fried sweet rice bread prepared during festivals), Kwati (a soup made from nine types of sprouted beans consumed during Janai Purnima), and various Newari delicacies such as Choila, Samay Baji, and Yomari.

Nepal's economy is structurally characterized by its status as a developing nation navigating the transitions from an agrarian base toward services and industrialization. As a landlocked country with exceptionally rugged terrain, Nepal faces inherent challenges, including high transit costs, infrastructural deficits, vulnerability to natural disasters, and a historical reliance on neighboring economies, particularly India, for international trade and transit routes.

Agriculture has traditionally formed the backbone of the Nepali economy, employing nearly two-thirds of the total workforce and contributing approximately 24 percent to the national Gross Domestic Product (GDP). However, agricultural productivity remains low due to a reliance on traditional farming techniques, fragmented land holdings, and heavy dependence on seasonal monsoon rains rather than modern irrigation networks. Principal food crops include rice, wheat, maize, millet, and barley, while cash crops like sugarcane, jute, tobacco, oilseeds, tea, coffee, and cardamom are grown for domestic processing and export. Animal husbandry is also vital, providing dairy, meat, and wool across the hills and mountains.

In recent decades, the service sector has expanded dramatically to become the largest contributor to Nepal's GDP. This growth is led by wholesale and retail trade, real estate, financial services, telecommunications, and tourism. Tourism is a vital source of foreign exchange and employment, leveraging the country's unparalleled natural and cultural assets. Millions of international visitors travel to Nepal annually for mountaineering, trekking in the Annapurna and Everest regions, wildlife safaris in the Terai, and cultural tours in the Kathmandu Valley.

A unique and defining structural feature of the modern Nepali economy is its heavy reliance on workers' remittances. Due to limited domestic employment opportunities, millions of young Nepalis travel abroad for employment, primarily to the Gulf Cooperation Council (GCC) countries (such as Malaysia, Qatar, Saudi Arabia, and the UAE) and to other nations worldwide. The inflow of formal remittances accounts for nearly 25 to 30 percent of the nation’s annual GDP, positioning Nepal among the top remittance-dependent countries globally. While these funds have been instrumental in reducing absolute poverty, improving household consumption, and maintaining foreign exchange reserves, they also point to a critical challenge: a "brain drain" and the loss of working-age labor needed for long-term domestic development.

The industrial sector in Nepal remains modest, contributing around 12 to 14 percent of GDP. Manufacturing activities are concentrated primarily in the processing of agricultural products, alongside light industries producing garments, carpets, pashmina shawls, cement, bricks, iron and steel, and beverages. Nepal's ready-made garments and hand-knotted woollen carpets were historically major export items, though they face stiff international competition today.

Hydropower represents the most promising long-term catalyst for Nepal's economic transformation. The country's steep topography and glacier-fed river systems provide a theoretical hydropower potential estimated at over 83,000 megawatts (MW), with about 42,000 MW considered technically and economically viable. For decades, inadequate investment, political instability, and policy bottlenecks meant that Nepal suffered from severe chronic electricity shortages (load shedding). However, strategic policy changes, private sector participation, and major cross-border transmission projects have allowed Nepal to achieve domestic electricity self-sufficiency and begin exporting surplus clean energy to India and Bangladesh during the wet summer months. Major ongoing projects, such as the Upper Tamakoshi, Arun III, and Upper Arun, are poised to transform the country into a key clean energy exporter in South Asia.

Nepal's international trade is characterized by a massive and growing trade deficit, as imports of petroleum products, machinery, vehicles, electronics, and luxury goods far outstrip the country's modest exports. India remains Nepal's largest trading partner by a wide margin, accounting for over 60 percent of total trade, facilitated by a porous open border and a currency peg between the Nepali Rupee (NPR) and the Indian Rupee (INR). China is the second-largest trading partner, with efforts underway to improve trans-Himalayan connectivity through roads and dry ports under infrastructure initiatives.

Tourism is a cornerstone of Nepal's global identity, serving as a dynamic bridge connecting the remote Himalayan nation with international travelers. Known widely as the "Roof of the World", Nepal offers an exceptional array of travel experiences, classified broadly into adventure tourism, cultural heritage tourism, wildlife tourism, and spiritual pilgrimage.

Adventure tourism is dominated by mountaineering and trekking. Since the historic first successful ascent of Mount Everest by Sir Edmund Hillary and Tenzing Norgay Sherpa on May 29, 1953, Nepal has been the ultimate destination for mountaineers. The country's Department of Tourism regulates access to hundreds of open mountain peaks, attracting elite climbers from around the globe during the spring and autumn climbing seasons. For non-mountaineers, trekking offers an immersive way to experience the high Himalayas. Iconic routes like the Everest Base Camp Trek, the Annapurna Circuit Trek, the Langtang Valley Trek, and the remote Upper Mustang Trek take travelers through breathtaking landscapes, rhododendron forests, alpine meadows, and traditional mountain villages, supported by a network of local teahouses. Beyond trekking, Nepal is an established hub for white-water rafting and kayaking along rivers like the Bhote Koshi and Trishuli, paragliding in the scenic skies of Pokhara, ultra-light aircraft flights, mountain biking, and bungee jumping over deep gorges.

For cultural heritage tourism, the Kathmandu Valley serves as a sprawling living museum. Within a radius of just a few kilometers, the valley hosts seven distinct groups of UNESCO World Heritage monuments: the Durbar Squares of Kathmandu (Hanuman Dhoka), Patan, and Bhaktapur; the sacred Hindu temples of Pashupatinath and Changu Narayan; and the majestic Buddhist stupas of Swayambhunath and Boudhanath. These sites display centuries of Newari artistic excellence, characterized by multilayered wooden pagodas, stone monoliths, and bronze icons that remain active centers of daily worship and community festivals. Outside the capital, the historic town of Janakpur attracts visitors to the majestic Janaki Temple, a masterpiece of bright Rajput-style architecture dedicated to Goddess Sita.

Wildlife and eco-tourism flourish in the southern Terai plains, proving that Nepal is much more than just snow-capped mountains. Chitwan National Park, another UNESCO World Heritage site, is globally celebrated for its successful conservation of the endangered Greater One-horned Rhinoceros and the Royal Bengal Tiger. Visitors explore the park via jeep safaris or guided canoe trips along the Rapti River, spotting wild elephants, sloth bears, leopards, and hundreds of bird species. Further west, Bardiya National Park offers an even more remote wilderness experience with high chances of tiger sightings, while the Koshi Tappu Wildlife Reserve in the east is a paradise for birdwatchers, hosting migratory birds from Siberia.

Spiritual and wellness tourism is an expanding sector. Lumbini, the birthplace of Buddha, stands as a universal sanctuary for peace, drawing millions of Buddhist pilgrims and spiritual seekers worldwide. The sacred Maya Devi Temple, the historic Ashoka Pillar, and the quiet monastic zones provide spaces for meditation and historical exploration. Additionally, Nepal has become a premier global destination for yoga retreats, meditation courses (such as Vipassana), and Ayurvedic wellness therapies, centered in the tranquil settings of Kathmandu, Pokhara, and rural hill stations.

Pokhara, often called the "tourism capital of Nepal," serves as the perfect base camp for adventures. Situated next to the calm waters of Fewa Lake and framed by the dramatic, fish-tailed peak of Mount Machapuchare and the Annapurna range, Pokhara offers a relaxed atmosphere, vibrant lakeside dining, and easy access to short hikes, boating, and adventure sports.

Nepal’s modern political architecture is defined by its transition to a Federal Democratic Republic, formalized under the landmark Constitution of Nepal adopted in 2015. This constitution marked the final chapter of a decade-long peace process following the Maoist insurgency, transforming a highly centralized, unitary state into a decentralized federal framework designed to bring governance closer to the people and address historical inequalities based on region, gender, and caste.

Under the federal structure, the government is organized into three distinct tiers: the Federal Government at the center, 7 Provinces, and 753 Local Governments (comprising metropolitan cities, sub-metropolitan cities, urban municipalities, and rural municipalities). Each tier possesses defined executive, legislative, and judicial powers, along with the authority to formulate budgets and collect specific taxes. The federal parliament is bicameral, consisting of the House of Representatives (Pratinidhi Sabha)—with members elected through a mix of direct first-past-the-post voting and proportional representation—and the National Assembly (Rastriya Sabha), representing the provinces. The President serves as the ceremonial Head of State, while executive governance is led by the Prime Minister, who must command a majority in the House of Representatives.

Administratively, the country's 7 provinces are numbered or named based on regional consensus (Koshi, Madhesh, Bagmati, Gandaki, Lumbini, Karnali, and Sudurpashchim). This federal restructuring aims to drive balanced regional development, ensuring that distant regions like the Far-West (Sudurpashchim) and Karnali receive equitable budget allocations and developmental focus, rather than resources concentrating solely in the Kathmandu Valley.

Despite these progressive frameworks, Nepal’s governance faces ongoing practical challenges. The country has a history of frequent political coalitions and leadership changes, which can slow the implementation of long-term development policies and affect institutional stability. Strengthening the administrative capacity of newly created local and provincial governments, ensuring clean governance, and curbing corruption remain top national priorities.

Looking ahead, Nepal faces a set of defining challenges and opportunities in the 21st century. The most pressing existential threat is climate change. Although Nepal contributes a negligible fraction of global greenhouse gas emissions, it is among the most vulnerable nations to climate impacts. Rapid melting of Himalayan glaciers, the formation of dangerous glacial lakes that threaten downstream communities with floods (GLOFs), unpredictable monsoon patterns, landslides in the hills, and severe droughts present direct threats to agriculture, hydropower, and human safety. Protecting this fragile mountain ecosystem requires international climate finance and regional cooperation.

Simultaneously, Nepal is working toward graduating from its status as a Least Developed Country (LDC) to a developing nation. To sustain this momentum, the government is prioritizing large-scale infrastructure development. This includes building major cross-border transport corridors, expanding international airports (such as Gautam Buddha International Airport in Bhairahawa and Pokhara International Airport), and accelerating strategic highway expansions like the Kathmandu-Terai Fast Track. By leveraging its strategic position between the economic giants of India and China, expanding its clean hydropower exports, and investing in human capital, Nepal aims to build a prosperous, resilient, and inclusive future for its people.

""") 

##### Splitting Loaded documents

In [153]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("data/text_files/data.txt")

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
	chunk_size = 1000,
	chunk_overlap = 150
)

chunks = splitter.split_documents(documents)
print("Documents:", len(documents))
print(f"Chunks:", len(chunks))

Documents: 1
Chunks: 44


In [157]:
for i, chunk in enumerate(chunks):
  print(f"---------Chunk {i+1}--------")
  # print(f"content:", chunk.page_content)
  print("Chunk:", chunk.metadata)

---------Chunk 1--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 2--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 3--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 4--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 5--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 6--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 7--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 8--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 9--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 10--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 11--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 12--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 13--------
Chunk: {'source': 'data/text_files/data.txt'}
---------Chunk 14--------
Chunk: {'source': 'data/text_files/data.txt'}
-

#### Experimentation on PDF

In [159]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("data/text_files/mypdf.pdf")

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
	chunk_size=1000,
	chunk_overlap=150
)

chunks = splitter.split_documents(documents)
for i, chunk in enumerate(chunks[:5]):
  print(f"\n--- Chunk {i+1} ---")
  print(chunk.page_content)
  print(chunk.metadata)


--- Chunk 1 ---
An ANPR-based Automatic Toll Tax Collection System 
using Camera 
 
B VeerasekharReddy  
Information Technology 
MLR Institute of Technology 
Hyderabad, India 
bhargavisekhar68@gmail.com 
 
Sahithi Sindhu Gadupudi 
Information Technology 
MLR Institute of Technology 
Hyderabad, India 
sahigadupudi@gmail.com 
 
Venkata Nagaraju Thatha 
Information Technology 
MLR Institute of Technology 
Hyderabad, India 
nagarajuthatha@gmail.com 
 
Sai Kumar Japala 
Information Technology 
MLR Institute of Technology 
Hyderabad, India 
j.saikumar174@gmail.com 
 
A Maanasa 
Information Technology  
MLR Institute of Technology 
Hyderabad, India 
ankireddymaanasa2105@gmail.com 
 
L Harshini Goud 
Information Technology  
MLR Institute of Technology 
Hyderabad, India 
harshinigoud16@gmail.com 
 
Abstract—Several toll collectio n systems have 
been developed in India for highways, such as 
Manual toll collection, RF  tags, Barcodes, and 
Number Plate Recognition. However, each system
{'prod